# TICKET MJD-503

De: Raquel Fernández · Jefa de ventas Cantabria
Para la campaña de reactivación: un mensaje personalizado para cada cliente de Cantabria que lleve
entre 90 y 180 días sin pedir, para que lo mande su comercial por WhatsApp. Con su nombre, cuánto lle‐
va sin pedir y el nombre del comercial.

entregables/mjd-503_mensajes.txt , un mensaje por cliente, separados por una línea en blanco.

In [9]:
import duckdb
from pathlib import Path

RAIZ = Path.home() / "master-data-ia"
BD = RAIZ / "db" / "marejada.duckdb"
print("BD:", BD, "| existe:", BD.exists())

con = duckdb.connect(str(BD), read_only=True)

clientes = con.execute("""
SELECT c.nombre_comercial,
c.comercial_asignado,
DATE '2026-09-30' - max(d.fecha) AS dias_sin_pedir
FROM gold.fact_ventas AS f
JOIN gold.dim_cliente AS c ON c.cliente_key = f.cliente_key
JOIN gold.dim_fecha AS d ON d.fecha_key = f.fecha_key
WHERE c.provincia = 'Cantabria' AND c.es_version_actual
GROUP BY c.cliente_id, c.nombre_comercial, c.comercial_asignado
HAVING dias_sin_pedir BETWEEN ? AND ?
ORDER BY dias_sin_pedir
""", [90, 180]).fetchall()
con.close()
print(f"{len(clientes)} clientes")
print(clientes[0])

BD: /Users/alvaroocrespoo/master-data-ia/db/marejada.duckdb | existe: True
18 clientes
('Supermercado El Botxo', 'Lorena Arce Díaz', 91)


In [11]:
def mensaje(cliente, comercial, dias):
    nombre_comercial = comercial.split()[0]
    semanas = dias // 7
    return (
        f"Hola, equipo de {cliente}. Soy {nombre_comercial}, de Marejada. "
        f"Hace {semanas} semanas que no os vemos por aquí y queríamos saber si todo va bien. "
        f"Si necesitáis algo para esta temporada, escribidme por aquí y os lo preparo."
    )

textos = [mensaje(nombre, comercial, dias) for nombre, comercial, dias in clientes]

salida = RAIZ / "entregables" / "mjd-503_mensajes.txt"
salida.parent.mkdir(parents=True, exist_ok=True)
with open(salida, "w", encoding="utf-8") as f:
    f.write("\n\n".join(textos) + "\n")

print(textos[0])

Hola, equipo de Supermercado El Botxo. Soy Lorena, de Marejada. Hace 13 semanas que no os vemos por aquí y queríamos saber si todo va bien. Si necesitáis algo para esta temporada, escribidme por aquí y os lo preparo.
